In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
from data_pipeline import load_and_prepare_data
import xgb_scripts

In [2]:
unique_frequencies = [
    2.54e-01, 3.40e-01, 4.56e-01, 6.12e-01, 8.22e-01, 9.99e-01, 1.10e+00, 1.33e+00,
    1.48e+00, 1.78e+00, 1.99e+00, 2.37e+00, 2.66e+00, 3.16e+00, 3.57e+00, 4.22e+00,
    4.80e+00, 5.62e+00, 6.43e+00, 7.50e+00, 8.64e+00, 1.00e+01, 1.16e+01, 1.33e+01,
    1.55e+01, 1.78e+01, 2.09e+01, 2.37e+01, 2.80e+01, 3.16e+01, 3.75e+01, 4.22e+01,
    5.03e+01, 5.62e+01, 6.76e+01, 7.50e+01, 9.06e+01, 1.02e+02, 1.22e+02, 1.35e+02,
    1.63e+02, 1.78e+02, 2.19e+02, 2.37e+02, 2.94e+02, 3.16e+02, 3.94e+02, 4.22e+02,
    5.29e+02, 5.64e+02, 7.10e+02, 7.50e+02, 9.52e+02, 1.00e+03, 1.28e+03, 1.33e+03,
    1.71e+03, 1.78e+03, 2.30e+03, 2.37e+03, 3.09e+03, 3.16e+03, 4.14e+03, 4.22e+03,
    5.56e+03, 5.62e+03, 7.45e+03, 7.50e+03, 1.00e+04
]

## Exploring relevant frequencies

## Systematic Frequency Selection Methodology

Following research best practices to determine optimal frequencies for our specific dataset.
This approach combines multiple statistical and physical criteria rather than just copying literature values.

## Comprehensive Feature Importance Analysis Using All Cycles Model

Loading the XGBoost model trained on ALL CYCLES (1-267) with binning + all frequencies to understand which specific frequencies and components are most important across the complete battery degradation spectrum - from early life through end-of-life conditions.

In [3]:
# Load the high-performing model trained on ALL CYCLES (1-267)
import joblib

models = joblib.load("../../models/xgb_binning_all_freq_all_cycles.pkl")
print(f"\nSuccessfully loaded XGBoost ensemble with {len(models)} models")
print(f"Model type: {type(models[0])}")


Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [4]:
# Load data with ALL CYCLES to match the saved model configuration
X_train, X_test, y_train, y_test = load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    method="bin_and_split",
)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"Train capacity range: {y_train.min():.1f} - {y_train.max():.1f} mAh")
print(f"Test capacity range: {y_test.min():.1f} - {y_test.max():.1f} mAh")
print(f"Data loaded: X_train shape = {X_train.shape}, X_test shape = {X_test.shape}")

# Get feature importance from the loaded models
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
print(f"Feature importance extracted from {len(models)} models")

# Sort features by importance
feature_indices = np.argsort(feature_importance)[::-1]  # Descending order
print(f"\nTop 10 most important features:")
for i in range(10):
    idx = feature_indices[i]
    print(f"{i+1:2d}. Feature {idx:3d}: Importance = {feature_importance[idx]:.4f}")

# Show cumulative importance
cumulative_importance = np.cumsum(feature_importance[feature_indices])
total_importance = np.sum(feature_importance)

print(f"\nCumulative importance breakdown:")
for n_features in [5, 10, 15, 20, 30]:
    if n_features <= len(feature_importance):
        pct = cumulative_importance[n_features-1] / total_importance * 100
        print(f"Top {n_features:2d} features capture {pct:.1f}% of total importance")

X_train: (290, 138), y_train: (290,)
X_test: (152, 138), y_test: (152,)
Train capacity range: 80.7 - 4070.0 mAh
Test capacity range: 78.0 - 3850.0 mAh
X_train: (290, 138), y_train: (290,)
X_test: (152, 138), y_test: (152,)
Train capacity range: 80.7 - 4070.0 mAh
Test capacity range: 78.0 - 3850.0 mAh
Data loaded: X_train shape = (290, 138), X_test shape = (152, 138)
Feature importance extracted from 10 models

Top 10 most important features:
 1. Feature  81: Importance = 0.4181
 2. Feature  48: Importance = 0.3639
 3. Feature  90: Importance = 0.0402
 4. Feature  18: Importance = 0.0327
 5. Feature  68: Importance = 0.0297
 6. Feature  66: Importance = 0.0149
 7. Feature   4: Importance = 0.0116
 8. Feature  64: Importance = 0.0100
 9. Feature  15: Importance = 0.0083
10. Feature  30: Importance = 0.0072

Cumulative importance breakdown:
Top  5 features capture 88.5% of total importance
Top 10 features capture 93.7% of total importance
Top 15 features capture 95.7% of total importance


In [ ]:
feature_details = []
for i in range(len(feature_importance)):
    if i < 69:  # Real impedance
        freq = unique_frequencies[i]
        feature_type = "Real"
    elif i < 138:  # Imaginary impedance  
        freq = unique_frequencies[i - 69]
        feature_type = "Imaginary"
    else:  # Action vector
        freq = None
        feature_type = "Action"
    
    feature_details.append({
        'idx': i,
        'type': feature_type,
        'frequency': freq,
        'importance': feature_importance[i]
    })

# Sort by importance (descending)
feature_details.sort(key=lambda x: x['importance'], reverse=True)

print("ALL FEATURES RANKED BY IMPORTANCE:")
print("Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total")

total_imp = sum(feature_importance)
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    if feature['frequency'] is not None:
        freq_str = f"{feature['frequency']:>8.2f}"
    else:
        freq_str = "   Action"
    
    print(f"{rank:4d} | {feature['idx']:7d} | {feature['type']:<9} | {freq_str} | {feature['importance']:10.4f} | {pct_individual:5.1f}% ({pct_cumulative:5.1f}%)")


ALL FEATURES RANKED BY IMPORTANCE:
Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total
   1 |      81 | Imaginary |     2.66 |     0.4181 |  41.8% ( 41.8%)
   2 |      48 | Real      |   529.00 |     0.3639 |  36.4% ( 78.2%)
   3 |      90 | Imaginary |    10.00 |     0.0402 |   4.0% ( 82.2%)
   4 |      18 | Real      |     6.43 |     0.0327 |   3.3% ( 85.5%)
   5 |      68 | Real      | 10000.00 |     0.0297 |   3.0% ( 88.5%)
   6 |      66 | Real      |  7450.00 |     0.0149 |   1.5% ( 89.9%)
   7 |       4 | Real      |     0.82 |     0.0116 |   1.2% ( 91.1%)
   8 |      64 | Real      |  5560.00 |     0.0100 |   1.0% ( 92.1%)
   9 |      15 | Real      |     4.22 |     0.0083 |   0.8% ( 92.9%)
  10 |      30 | Real      |    37.50 |     0.0072 |   0.7% ( 93.7%)
  11 |      67 | Real      |  7500.00 |     0.0059 |   0.6% ( 94.2%)
  12 |     132 | Imaginary |  4220.00 |     0.0053 |   0.5% ( 94.8%)
  13 |      20 | Real      |     8.64 |     0.0035 |   0.4% ( 95.1%

In [6]:
# Debug: Why are Action Vector features having no effect?
print("=== ACTION VECTOR DEBUGGING ===\n")

# Check if Action Vector features are even in the feature set
total_features = len(feature_importance)
print(f"Total features: {total_features}")
print(f"Expected: 69 Real + 69 Imaginary + 2 Action = 140 features")

if total_features < 140:
    print("⚠️  ACTION VECTOR MISSING: Features < 140, action vector not included!")
else:
    print("✓ Action vector features should be present")

# Check action vector feature importance specifically
action_feature_indices = [i for i in range(138, total_features)]  # Features 138+ are action vector
print(f"\nAction vector feature indices: {action_feature_indices}")

if action_feature_indices:
    print(f"Action vector feature importances:")
    for idx in action_feature_indices:
        importance = feature_importance[idx]
        pct = (importance / total_imp) * 100
        print(f"  Feature {idx}: {importance:.6f} ({pct:.3f}%)")
    
    # Find action vector rank in overall importance
    action_ranks = []
    for idx in action_feature_indices:
        rank = next((i for i, feat in enumerate(feature_details, 1) if feat['idx'] == idx), None)
        action_ranks.append(rank)
    
    print(f"\nAction vector feature ranks: {action_ranks}")
    print(f"Best action vector rank: {min(action_ranks) if action_ranks else 'N/A'}")
else:
    print("❌ No action vector features found!")

# Let's also check what the actual action vector data looks like
print(f"\n=== ACTION VECTOR DATA INSPECTION ===")
from feature_engineering.action_vector import build_action_vector
import pandas as pd

# Load sample data
df_sample = pd.read_csv("../../data/04-03-24/A1.csv")
cycle_nums, action_matrix = build_action_vector(df_sample)

print(f"Action matrix shape: {action_matrix.shape}")
print(f"Action matrix stats:")
print(f"  Charge current: min={action_matrix[:,0].min():.3f}, max={action_matrix[:,0].max():.3f}, mean={action_matrix[:,0].mean():.3f}")
print(f"  Discharge current: min={action_matrix[:,1].min():.3f}, max={action_matrix[:,1].max():.3f}, mean={action_matrix[:,1].mean():.3f}")

# Check for issues
nan_charge = pd.isna(action_matrix[:,0]).sum()
nan_discharge = pd.isna(action_matrix[:,1]).sum()
print(f"  NaN values: charge={nan_charge}, discharge={nan_discharge}")

# Compare scales with impedance features
print(f"\n=== FEATURE SCALE COMPARISON ===")
print(f"Sample impedance values from training data:")
print(f"  Real impedance range: {X_train[:, :69].min():.3f} to {X_train[:, :69].max():.3f}")
print(f"  Imaginary impedance range: {X_train[:, 69:138].min():.3f} to {X_train[:, 69:138].max():.3f}")

if total_features >= 140:
    print(f"  Action vector range: {X_train[:, 138:].min():.3f} to {X_train[:, 138:].max():.3f}")
    
    # Scale difference analysis
    impedance_scale = X_train[:, :138].std()
    action_scale = X_train[:, 138:].std()
    scale_ratio = impedance_scale / action_scale if action_scale > 0 else float('inf')
    print(f"  Scale ratio (impedance/action): {scale_ratio:.1f}")
    
    if scale_ratio > 1000:
        print("  ⚠️  SCALE MISMATCH: Action vector much smaller scale than impedance!")
    elif scale_ratio < 0.001:
        print("  ⚠️  SCALE MISMATCH: Action vector much larger scale than impedance!")
    else:
        print("  ✓ Scales are reasonably comparable")
else:
    print("  Action vector features not found in training data")

=== ACTION VECTOR DEBUGGING ===

Total features: 140
Expected: 69 Real + 69 Imaginary + 2 Action = 140 features
✓ Action vector features should be present

Action vector feature indices: [138, 139]
Action vector feature importances:
  Feature 138: 0.000000 (0.000%)
  Feature 139: 0.000000 (0.000%)

Action vector feature ranks: [139, 140]
Best action vector rank: 139

=== ACTION VECTOR DATA INSPECTION ===
Action matrix shape: (270, 2)
Action matrix stats:
  Charge current: min=nan, max=nan, mean=nan
  Discharge current: min=nan, max=nan, mean=nan
  NaN values: charge=1, discharge=2

=== FEATURE SCALE COMPARISON ===
Sample impedance values from training data:
  Real impedance range: 0.000 to 1.000
  Imaginary impedance range: 0.000 to 1.000
  Action vector range: 7990.000 to 15000.000
  Scale ratio (impedance/action): 0.0
  ⚠️  SCALE MISMATCH: Action vector much larger scale than impedance!
Action matrix shape: (270, 2)
Action matrix stats:
  Charge current: min=nan, max=nan, mean=nan
  